In [ ]:
import torch
import torchvision
import numpy as np
import torch.nn as nn
import time
import os
import json
import subprocess
import threading
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
import matplotlib.pyplot as plt
import io
from openpyxl.drawing.image import Image as XLImage
from torchvision.transforms import v2
from bayesian_torch.models.dnn_to_bnn import dnn_to_bnn
from medmnist import PathMNIST, INFO

# Hyperparamètres
batch_size      = 128
NUM_MONTE_CARLO = 100
N_REPS_VALUES   = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

CONFIGS = [
    ("BayMoped_full",  None, True,  True),
    ("BayMoped_n6",   6,    True,  True),
    ("BayMoped_n32",  32,   True,  True),
    ("EffNetBase",    None, False, False),
    ("BayNoMoped",    None, False, True),
    ("BayMoped_n1",   1,    True,  True),
    ("BayMoped_n19",  19,   True,  True),
    ("BayMoped_n162", 162,  True,  True),
    ("BayMoped_n201", 201,  True,  True),
]

# Dataset
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
info       = INFO['pathmnist']
testset    = PathMNIST(split="test", download=True, size=28, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                          shuffle=False, num_workers=4, pin_memory=True)
device = torch.device(torch.accelerator.current_accelerator().type
                      if torch.accelerator.is_available() else 'cpu')

# GPU monitoring
class GPUMonitor:
    def __init__(self, interval=0.5):
        self.interval = interval
        self.powers = []; self.temps = []
        self._running = False; self._thread = None

    def start(self):
        self.powers = []; self.temps = []
        self._running = True
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()

    def stop(self):
        self._running = False
        if self._thread: self._thread.join()

    def _run(self):
        while self._running:
            try:
                out = subprocess.check_output(
                    ["nvidia-smi", "--query-gpu=power.draw,temperature.gpu",
                     "--format=csv,noheader,nounits"],
                    stderr=subprocess.DEVNULL
                ).decode().strip().split(",")
                self.powers.append(float(out[0]))
                self.temps.append(float(out[1]))
            except Exception:
                pass
            time.sleep(self.interval)

    def stats(self):
        p = self.powers; t = self.temps
        return {
            'mean_power': float(np.mean(p)) if p else None,
            'std_power':  float(np.std(p))  if p else None,
            'mean_temp':  float(np.mean(t)) if t else None,
            'std_temp':   float(np.std(t))  if t else None,
        }

# Utilitaires
def split_model_graph(model, n):
    traced = torch.fx.symbolic_trace(model)
    nodes  = list(traced.graph.nodes)
    module_nodes = [nd for nd in nodes if nd.op == 'call_module']
    if n >= len(module_nodes):
        raise ValueError(f"Cannot split off {n} layers.")
    first_b = module_nodes[-n]
    cut_idx = nodes.index(first_b)
    nodes_A = nodes[:cut_idx]; nodes_B = nodes[cut_idx:]
    boundary = []
    for nd in nodes_B:
        for inp in nd.all_input_nodes:
            if inp in nodes_A and inp not in boundary:
                boundary.append(inp)
    graph_A, env_A = torch.fx.Graph(), {}
    for nd in nodes_A:
        env_A[nd] = graph_A.node_copy(nd, lambda x: env_A[x])
    out_args = tuple(env_A[nd] for nd in boundary)
    graph_A.output(out_args[0] if len(out_args) == 1 else out_args)
    model_A = torch.fx.GraphModule(traced, graph_A)
    graph_B, env_B = torch.fx.Graph(), {}
    for nd in boundary:
        env_B[nd] = graph_B.placeholder(nd.name)
    for nd in nodes_B:
        env_B[nd] = graph_B.node_copy(nd, lambda x: env_B[x])
    model_B = torch.fx.GraphModule(traced, graph_B)
    return model_A, model_B

def build_model(n_couches, moped, is_bayesian):
    net = torchvision.models.efficientnet_b0(progress=True)
    net.classifier[1] = nn.Linear(1280, 9)
    red = torch.load("effNet.pth", map_location=device)
    net.load_state_dict(red['model_state_dict'])
    const_bnn = {
        "prior_mu": 0.0, "prior_sigma": 1.0,
        "posterior_mu_init": 0.0, "posterior_rho_init": -3.0,
        "type": "Reparameterization",
        "moped_enable": moped, "moped_delta": 0.5,
    }
    if not is_bayesian:
        net.to(device); return net, None
    if n_couches is None:
        dnn_to_bnn(net, const_bnn); net.to(device); return net, None
    model_A, model_B = split_model_graph(net, n=n_couches)
    dnn_to_bnn(model_B, const_bnn)
    model_A.to(device).eval()
    for p in model_A.parameters(): p.requires_grad = False
    model_B.to(device)
    return model_B, model_A

def evaluate_once(model, model_A):
    """One complete evaluation — returns acc, time, gpu stats."""
    model.eval()
    all_preds, all_labels = [], []
    gpu = GPUMonitor(); gpu.start()
    t0  = time.time()

    with torch.no_grad():
        for data in testloader:
            inputs, labels = data[0].to(device), data[1].to(device)
            labels = labels.squeeze(1)
            output_mc = []
            for _ in range(NUM_MONTE_CARLO):
                logits = model(model_A(inputs)) if model_A is not None else model(inputs)
                output_mc.append(torch.nn.functional.softmax(logits, dim=-1))
            output = torch.stack(output_mc)
            all_preds.append(torch.argmax(output.mean(dim=0), dim=1).cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    elapsed = time.time() - t0
    gpu.stop()

    all_preds  = np.concatenate(all_preds,  axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    acc = float((all_preds == all_labels).mean())

    return {'acc': acc, 'time': elapsed, **gpu.stats()}

# JSON helpers
def load_json(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return {}

def save_json(path, data):
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)

# Excel graphiques
def make_chart_buf(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=120)
    plt.close(fig); buf.seek(0); return buf

def update_excel(config_name, all_runs_data, filename="res_mc.xlsx"):
    """
    all_runs_data : dict { run_id (str) -> list of 100 eval dicts }
    generate summary and graphs in Excel for the given config.
    """
    hf    = Font(bold=True, color="FFFFFF", name="Arial")
    hfill = PatternFill("solid", start_color="2F4F8F")
    nf    = Font(name="Arial")
    mf    = PatternFill("solid", start_color="E8F4FD")
    sf    = PatternFill("solid", start_color="FDF3E8")

    def sh(cell, v):
        cell.value = v; cell.font = hf; cell.fill = hfill
        cell.alignment = Alignment(horizontal="center")
    def cw(ws, col, w):
        ws.column_dimensions[openpyxl.utils.get_column_letter(col)].width = w
    def r(v, d=2):
        return round(v, d) if v is not None else 'N/A'

    wb = openpyxl.load_workbook(filename) if os.path.exists(filename) else openpyxl.Workbook()
    if wb.active.title == "Sheet": wb.active.title = "Summary"

    # Summary
    if "Summary" not in wb.sheetnames:
        ws_s = wb.create_sheet("Summary", 0)
    else:
        ws_s = wb["Summary"]

    s_hdrs = ["Config", "N Reps",
              "Mean Acc (%)", "Std Acc (%)",
              "Mean Time (s)", "Std Time (s)",
              "Mean Power (W)", "Std Power (W)",
              "Mean Temp (°C)", "Std Temp (°C)"]
    if ws_s.cell(1, 1).value is None:
        for col, h in enumerate(s_hdrs, start=1):
            sh(ws_s.cell(1, col), h); cw(ws_s, col, 18)

    # Delete previous rows for this config
    rows_to_del = [i for i, row in enumerate(ws_s.iter_rows(min_row=2, values_only=True), start=2)
                   if row[0] == config_name]
    for i in reversed(rows_to_del):
        ws_s.delete_rows(i)

    # Add new rows for this config
    for n_reps in N_REPS_VALUES:
        all_evals = []
        for run_evals in all_runs_data.values():
            all_evals.extend(run_evals[:n_reps])
        if not all_evals: continue

        accs  = [e['acc']  for e in all_evals]
        times = [e['time'] for e in all_evals]
        pwrs  = [e['mean_power'] for e in all_evals if e['mean_power'] is not None]
        stds_p= [e['std_power']  for e in all_evals if e['std_power']  is not None]
        tmps  = [e['mean_temp']  for e in all_evals if e['mean_temp']  is not None]
        stds_t= [e['std_temp']   for e in all_evals if e['std_temp']   is not None]

        row = ws_s.max_row + 1
        for col, v in enumerate([
            config_name, n_reps,
            r(100*np.mean(accs)),  r(100*np.std(accs)),
            r(np.mean(times)),     r(np.std(times)),
            r(np.mean(pwrs))  if pwrs  else 'N/A', r(np.mean(stds_p)) if stds_p else 'N/A',
            r(np.mean(tmps))  if tmps  else 'N/A', r(np.mean(stds_t)) if stds_t else 'N/A',
        ], start=1):
            ws_s.cell(row, col, v).font = nf

    # Config detail sheet
    sheet_name = config_name[:31]
    if sheet_name in wb.sheetnames:
        del wb[sheet_name]
    ws2 = wb.create_sheet(sheet_name)

    # Data for graphs
    mean_accs, std_accs, mean_times, std_times = [], [], [], []
    mean_pwrs, std_pwrs, mean_tmps, std_tmps   = [], [], [], []

    for n_reps in N_REPS_VALUES:
        all_evals = []
        for run_evals in all_runs_data.values():
            all_evals.extend(run_evals[:n_reps])

        accs  = [e['acc']  for e in all_evals]
        times = [e['time'] for e in all_evals]
        pwrs  = [e['mean_power'] for e in all_evals if e['mean_power'] is not None]
        tmps  = [e['mean_temp']  for e in all_evals if e['mean_temp']  is not None]

        mean_accs.append(100*np.mean(accs));  std_accs.append(100*np.std(accs))
        mean_times.append(np.mean(times));    std_times.append(np.std(times))
        mean_pwrs.append(np.mean(pwrs) if pwrs else None)
        std_pwrs.append(np.std(pwrs)   if pwrs else None)
        mean_tmps.append(np.mean(tmps) if tmps else None)
        std_tmps.append(np.std(tmps)   if tmps else None)

    # Graphique 1 : Acc mean ± std vs N_Reps
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(N_REPS_VALUES, mean_accs, yerr=std_accs,
                marker='o', color='steelblue', capsize=4, label="Acc ± std")
    ax.set_xlabel("N Repetitions"); ax.set_ylabel("Accuracy (%)")
    ax.set_title(f"{config_name} — Accuracy vs N Reps")
    ax.grid(True, alpha=0.3); ax.legend()
    img = XLImage(make_chart_buf(fig)); img.anchor = "A1"; ws2.add_image(img)

    # Graphique 2 : Std acc vs N_Reps (stability)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(N_REPS_VALUES, std_accs, marker='o', color='steelblue')
    ax.set_xlabel("N Repetitions"); ax.set_ylabel("Std Accuracy (%)")
    ax.set_title(f"{config_name} — Stability vs N Reps")
    ax.grid(True, alpha=0.3)
    img = XLImage(make_chart_buf(fig)); img.anchor = "J1"; ws2.add_image(img)

    # Graphique 3 : Time vs N_Reps
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(N_REPS_VALUES, mean_times, yerr=std_times,
                marker='o', color='orange', capsize=4)
    ax.set_xlabel("N Repetitions"); ax.set_ylabel("Time (s)")
    ax.set_title(f"{config_name} — Time vs N Reps")
    ax.grid(True, alpha=0.3)
    img = XLImage(make_chart_buf(fig)); img.anchor = "A25"; ws2.add_image(img)

    # Graphique 4 : Power vs N_Reps
    if any(v is not None for v in mean_pwrs):
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.errorbar(N_REPS_VALUES, mean_pwrs, yerr=std_pwrs,
                    marker='o', color='red', capsize=4)
        ax.set_xlabel("N Repetitions"); ax.set_ylabel("Power (W)")
        ax.set_title(f"{config_name} — GPU Power vs N Reps")
        ax.grid(True, alpha=0.3)
        img = XLImage(make_chart_buf(fig)); img.anchor = "J25"; ws2.add_image(img)

    # Graphique 5 : Temp vs N_Reps
    if any(v is not None for v in mean_tmps):
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.errorbar(N_REPS_VALUES, mean_tmps, yerr=std_tmps,
                    marker='o', color='green', capsize=4)
        ax.set_xlabel("N Repetitions"); ax.set_ylabel("Temperature (°C)")
        ax.set_title(f"{config_name} — GPU Temp vs N Reps")
        ax.grid(True, alpha=0.3)
        img = XLImage(make_chart_buf(fig)); img.anchor = "A49"; ws2.add_image(img)

    wb.save(filename)
    print(f"  Excel mis à jour : {sheet_name}")

# Main loop
def run_mc_stability():
    for config_name, n_couches, moped, is_bayesian in CONFIGS:
        print(f"\n{'='*55}")
        print(f"Config : {config_name}")
        print(f"{'='*55}")

        json_path = f"{config_name}_mc.json"
        all_runs_data = load_json(json_path)

        for run_i in range(10):
            run_key = f"run_{run_i}"
            if run_key in all_runs_data:
                print(f"  Run {run_i} déjà évalué, skip.")
                continue

            save_path = f"{config_name}_run{run_i}.pth"
            if not os.path.exists(save_path):
                print(f"  Fichier manquant : {save_path}, skip.")
                continue

            print(f"  Run {run_i} — chargement...")
            checkpoint = torch.load(save_path, map_location=device)
            model, model_A = build_model(n_couches, moped, is_bayesian)
            model.load_state_dict(checkpoint['model_state_dict'])
            if model_A is not None and checkpoint.get('model_A_state_dict'):
                model_A.load_state_dict(checkpoint['model_A_state_dict'])

            print(f"  Run {run_i} — évaluation {max(N_REPS_VALUES)} fois (MC={NUM_MONTE_CARLO})...")
            run_evals = []
            for rep_i in range(max(N_REPS_VALUES)):
                result = evaluate_once(model, model_A)
                run_evals.append(result)
                if (rep_i + 1) % 10 == 0:
                    print(f"    {rep_i+1}/{max(N_REPS_VALUES)} — acc: {100*result['acc']:.2f}%")

            # Sauvegarder dans JSON et mettre à jour Excel
            all_runs_data[run_key] = run_evals
            save_json(json_path, all_runs_data)
            update_excel(config_name, all_runs_data, filename="res_mc.xlsx")
            print(f"  Run {run_i} sauvegardé.")

run_mc_stability()


Config : BayMoped_full
  Run 0 déjà évalué, skip.
  Run 1 déjà évalué, skip.
  Run 2 déjà évalué, skip.
  Run 3 — chargement...
  Run 3 — évaluation 100 fois (MC=100)...
    10/100 — acc: 84.07%
    20/100 — acc: 83.96%
    30/100 — acc: 83.91%
    40/100 — acc: 83.83%
    50/100 — acc: 83.75%
    60/100 — acc: 83.91%
    70/100 — acc: 83.84%
    80/100 — acc: 83.73%
    90/100 — acc: 83.90%
    100/100 — acc: 84.12%
  Excel mis à jour : BayMoped_full
  Run 3 sauvegardé.
  Run 4 — chargement...
  Run 4 — évaluation 100 fois (MC=100)...
    10/100 — acc: 82.12%
    20/100 — acc: 82.09%
    30/100 — acc: 81.92%
    40/100 — acc: 82.20%
    50/100 — acc: 82.19%
    60/100 — acc: 82.26%
    70/100 — acc: 82.05%
    80/100 — acc: 82.26%
    90/100 — acc: 81.89%
    100/100 — acc: 82.06%
  Excel mis à jour : BayMoped_full
  Run 4 sauvegardé.
  Run 5 — chargement...
  Run 5 — évaluation 100 fois (MC=100)...
    10/100 — acc: 81.81%
    20/100 — acc: 81.74%
    30/100 — acc: 81.80%
    40/100

In [6]:
def generate_final_excel(filename="res_mc_final.xlsx"):
    for config_name, n_couches, moped, is_bayesian in CONFIGS:
        json_path = f"{config_name}_mc.json"
        if not os.path.exists(json_path):
            print(f"JSON manquant : {json_path}, skip.")
            continue
        all_runs_data = load_json(json_path)
        n_runs = len(all_runs_data)
        print(f"{config_name} : {n_runs} runs trouvés")
        update_excel(config_name, all_runs_data, filename=filename)

generate_final_excel()

BayMoped_full : 3 runs trouvés
  Excel mis à jour : BayMoped_full
BayMoped_n6 : 10 runs trouvés
  Excel mis à jour : BayMoped_n6
BayMoped_n32 : 10 runs trouvés
  Excel mis à jour : BayMoped_n32
EffNetBase : 10 runs trouvés
  Excel mis à jour : EffNetBase
BayNoMoped : 10 runs trouvés
  Excel mis à jour : BayNoMoped
BayMoped_n1 : 10 runs trouvés
  Excel mis à jour : BayMoped_n1
BayMoped_n19 : 10 runs trouvés
  Excel mis à jour : BayMoped_n19
BayMoped_n162 : 10 runs trouvés
  Excel mis à jour : BayMoped_n162
BayMoped_n201 : 10 runs trouvés
  Excel mis à jour : BayMoped_n201


In [16]:
import torch
import torchvision
import torch.nn as nn
from bayesian_torch.models.dnn_to_bnn import dnn_to_bnn
from medmnist import PathMNIST
from torchvision.transforms import v2
import numpy as np
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Dataset ────────────────────────────────────────────────────
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
testset = PathMNIST(split="test", download=True, size=28, transform=transform)
N_IMAGES = 30
indices = torch.randperm(len(testset))[:N_IMAGES].tolist()
images  = torch.stack([testset[i][0] for i in indices]).to(device)
labels  = [int(testset[i][1][0]) for i in indices]

# ── Modèle ─────────────────────────────────────────────────────
CHECKPOINT = "BayMoped_full_run1.pth"  # <- change si besoin

net = torchvision.models.efficientnet_b0(progress=False)
net.classifier[1] = nn.Linear(1280, 9)
checkpoint = torch.load(CHECKPOINT, map_location=device)
const_bnn = {
    "prior_mu": 0.0, "prior_sigma": 1.0,
    "posterior_mu_init": 0.0, "posterior_rho_init": -3.0,
    "type": "Reparameterization",
    "moped_enable": True, "moped_delta": 0.5,
}
dnn_to_bnn(net, const_bnn)
net.load_state_dict(checkpoint['model_state_dict'])
net.to(device).eval()

# ── 100 reps × 100 passes MC sur 20 images ────────────────────
NUM_REPS = 100
NUM_MC   = 100

# all_preds[img_idx][rep] = classe prédite (vote majoritaire sur 100 passes MC)
all_preds = [[] for _ in range(N_IMAGES)]
# all_mc_votes[img_idx][rep] = liste brute des 100 votes MC pour cette rep
all_mc_votes = [[] for _ in range(N_IMAGES)]

with torch.no_grad():
    for rep in range(NUM_REPS):
        mc_votes = [[] for _ in range(N_IMAGES)]
        for _ in range(NUM_MC):
            logits = net(images)
            for img_i, pred in enumerate(logits.argmax(dim=1).tolist()):
                mc_votes[img_i].append(pred)
        for img_i in range(N_IMAGES):
            vote = max(set(mc_votes[img_i]), key=mc_votes[img_i].count)
            all_preds[img_i].append(vote)
            all_mc_votes[img_i].append(mc_votes[img_i])
        if (rep + 1) % 10 == 0:
            print(f"Rep {rep+1}/{NUM_REPS} done")



# ── Construction de l'Excel ────────────────────────────────────
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "MC Stability"

hf      = Font(bold=True, color="FFFFFF", name="Arial", size=11)
hfill   = PatternFill("solid", start_color="2F4F8F")
nf      = Font(name="Arial", size=10)
nf_bold = Font(name="Arial", size=10, bold=True)
stable_fill   = PatternFill("solid", start_color="D9EAD3")  # vert clair
unstable_fill = PatternFill("solid", start_color="F4CCCC")  # rouge clair
correct_font  = Font(name="Arial", size=10, color="2E7D32")
wrong_font    = Font(name="Arial", size=10, color="C62828")
thin = Side(style="thin", color="CCCCCC")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

def sh(cell, v):
    cell.value = v
    cell.font = hf
    cell.fill = hfill
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = border

def cw(col, w):
    ws.column_dimensions[get_column_letter(col)].width = w

# Titre
ws.merge_cells("A1:F1")
ws["A1"] = f"MC Prediction Stability — {CHECKPOINT}"
ws["A1"].font = Font(bold=True, size=14, name="Arial")
ws["A1"].alignment = Alignment(horizontal="left")

ws.merge_cells("A2:F2")
ws["A2"] = f"{N_IMAGES} random test images · {NUM_MC} MC passes × {NUM_REPS} repetitions"
ws["A2"].font = Font(italic=True, size=9, color="666666", name="Arial")

# ── Tableau : répartition des votes sur 100 reps ───────────────
start_row = 4
headers = ["Image #", "True Label", "Majority Vote", "Vote Breakdown (/100 reps)", "Distinct Classes", "Stability"]
for col, h in enumerate(headers, start=1):
    sh(ws.cell(start_row, col), h)
cw(1, 10); cw(2, 12); cw(3, 14); cw(4, 40); cw(5, 16); cw(6, 12)

for img_i in range(N_IMAGES):
    row = start_row + 1 + img_i
    preds = all_preds[img_i]  # 100 votes (1 par rep)

    ws.cell(row, 1, img_i).font = nf
    ws.cell(row, 1).alignment = Alignment(horizontal="center")
    ws.cell(row, 1).border = border

    ws.cell(row, 2, labels[img_i]).font = nf_bold
    ws.cell(row, 2).alignment = Alignment(horizontal="center")
    ws.cell(row, 2).border = border

    majority = max(set(preds), key=preds.count)
    c_maj = ws.cell(row, 3, majority)
    c_maj.alignment = Alignment(horizontal="center")
    c_maj.border = border
    c_maj.font = correct_font if majority == labels[img_i] else wrong_font

    counts = {cls: preds.count(cls) for cls in sorted(set(preds), key=lambda c: -preds.count(c))}
    breakdown = " | ".join(f"{cls}: {n}" for cls, n in counts.items())
    c_brk = ws.cell(row, 4, breakdown)
    c_brk.alignment = Alignment(horizontal="left")
    c_brk.border = border
    c_brk.font = nf

    n_distinct = len(counts)
    c_dist = ws.cell(row, 5, n_distinct)
    c_dist.alignment = Alignment(horizontal="center")
    c_dist.border = border
    c_dist.font = nf_bold if n_distinct == 1 else Font(name="Arial", size=10, bold=True, color="C62828")

    c_stab = ws.cell(row, 6)
    c_stab.border = border
    c_stab.alignment = Alignment(horizontal="center")
    if n_distinct == 1:
        c_stab.value = "Stable"
        c_stab.fill = stable_fill
        c_stab.font = Font(name="Arial", size=10, color="2E7D32", bold=True)
    else:
        c_stab.value = "Varies"
        c_stab.fill = unstable_fill
        c_stab.font = Font(name="Arial", size=10, color="C62828", bold=True)

# ── Résumé global ────────────────────────────────────────────
summary_row = start_row + N_IMAGES + 3
ws.merge_cells(f"A{summary_row}:D{summary_row}")
ws[f"A{summary_row}"] = "Summary"
ws[f"A{summary_row}"].font = Font(bold=True, size=12, name="Arial")

n_stable_imgs = sum(1 for img_i in range(N_IMAGES) if len(set(all_preds[img_i])) == 1)
n_correct_final = sum(
    1 for img_i in range(N_IMAGES)
    if max(set(all_preds[img_i]), key=all_preds[img_i].count) == labels[img_i]
)

summary_data = [
    ("Total images tested", N_IMAGES),
    ("Images with stable prediction (no class change across reps)", n_stable_imgs),
    ("Images with varying prediction across reps", N_IMAGES - n_stable_imgs),
    ("Final accuracy (n=100, majority vote)", f"{100 * n_correct_final / N_IMAGES:.1f}%"),
]
for i, (label, val) in enumerate(summary_data):
    r = summary_row + 1 + i
    ws.cell(r, 1, label).font = nf
    ws.cell(r, 4, val).font = nf_bold
    ws.cell(r, 4).alignment = Alignment(horizontal="center")

ws.freeze_panes = f"C{start_row + 1}"

wb.save("mc_stability_report.xlsx")
print("\nExcel généré : mc_stability_report.xlsx")

Rep 10/100 done
Rep 20/100 done
Rep 30/100 done
Rep 40/100 done
Rep 50/100 done
Rep 60/100 done
Rep 70/100 done
Rep 80/100 done
Rep 90/100 done
Rep 100/100 done

Excel généré : mc_stability_report.xlsx
